In [ ]:
from sqlalchemy import create_engine

import folium
import math
import numpy as np
import pandas as pd
import pymysql
import rdflib as rdf

In [ ]:
engine = create_engine(open("conn.txt", "r").read())

In [ ]:
# get vocabulary items
q =  "SELECT * FROM vocabulary"
vocabs_os_df = pd.read_sql_query(q, engine)
vocabs_os_df.set_index('id', inplace=True)

q = "SELECT * FROM property"
props_os_df = pd.read_sql_query(q, engine).merge(vocabs_os_df, left_on = 'vocabulary_id', right_on='id')

props_os_df['predicate'] = props_os_df[['namespace_uri', 'local_name']].apply(lambda x: ''.join(x), axis=1)

print(f'Length of df:{len(props_os_df)}')

In [ ]:
q = "SELECT * FROM value"

items_tmp_df = pd.read_sql_query(q, engine).merge(props_os_df, left_on = 'property_id', right_on = 'id',
                                                suffixes=('_value', '_vocab'))

print(f'Length of df:{len(items_tmp_df)}')

In [ ]:
items_df_strings = items_tmp_df[['resource_id','predicate','value_resource_id','value', 'uri']].merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
   left_on = 'resource_id', right_on = 'resource_id', how = 'left')

items_df_strings.rename(columns={'value_y':'subject', 'value_x':'object_str'},inplace=True)
items_df_strings.drop(columns='resource_id', inplace=True)

print(f'Length of df:{len(items_df_strings)}')

In [ ]:
# optional further bit of checking
#items_df_strings.predicate.unique()

In [ ]:
items_df_things = items_df_strings.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
   left_on = 'value_resource_id', right_on = 'resource_id', how = 'left')

items_df_things.rename(columns={'value':'object_thing'},inplace=True)
items_df_things.drop(columns='value_resource_id', inplace=True)

items_df = items_df_things[['subject','predicate', 'object_thing', 'object_str', 'uri']]

print(f'Length of df:{len(items_df)}')

# Should evaluate to True. Side effects of how Omeka S treats Item Sets, etc.
print(len(items_df[items_df.subject.isnull()]) == 4)

In [ ]:
# keep the rows with subject that are not null.
items_df = items_df[items_df.subject.notnull()]
print(f'Length of df:{len(items_df)}')

In [ ]:
G = rdf.Graph()

In [ ]:
for i,r in items_df.iterrows():
    if str((r['object_thing'])) != 'nan':
        G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.object_thing}')))
    elif r['uri'] != None:
        if r['object_str'] != None:
            G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(r.uri)))
        elif r['object_str'] == None:
            G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(r.uri)))
    elif r['object_str'] != None:
        G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.Literal(r.object_str)))

In [ ]:
# in progress
q = "SELECT * FROM mapping_marker"

df = pd.read_sql_query(q, engine)

mm_df = df.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
            left_on = 'item_id',
            right_on = 'resource_id', how = 'left')\
            [['value','geojson']].rename(columns = {'value':'subject'})

In [ ]:
for i,r in mm_df.iterrows():
    G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(u'http://p-lod.umasscreate.net/vocabulary#geojson'),
               rdf.Literal(r.geojson)))

In [ ]:
# in progress
q = "SELECT * FROM resource WHERE resource_class_id = 151"

df = pd.read_sql_query(q, engine)

class_df = df.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
            left_on = 'id',
            right_on = 'resource_id', how = 'left')\
            ['value'].rename(columns = {'value':'subject'})

In [ ]:
for r in class_df:
    G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r}'),
               rdf.URIRef(u'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
               rdf.URIRef(u'http://www.w3.org/2002/07/owl#Class')))

In [ ]:
ns_dict = dict(zip(vocabs_os_df.prefix,vocabs_os_df.namespace_uri))

for k in ns_dict.keys():
    G.bind(k,ns_dict[k])

In [ ]:
ttl = G.serialize(destination = None, format="turtle").decode()
with open("p-lod.ttl", "w") as file:
    file.write(ttl)

In [ ]:
%time
# confirm that .ttl file can be parsed by rdflib. seems like a good idea.
G = rdf.Graph()
G.parse("p-lod.ttl", format = "turtle")

In [ ]:
%time
# and now query it
result = G.query("SELECT ?s ?o WHERE {?s <http://p-lod.umasscreate.net/vocabulary#geojson> ?o}")
print(len(result))

In [ ]:
%time
#time getting all triples and print how many. useful info.
result = G.query("SELECT * WHERE {?s ?p ?o}")
print(len(result))